# 🌧️ Pipeline CEMADEN: Bronze & Prata

Notebook operacional para disparar a ingestão dos dados pluviométricos brutos do CEMADEN (Bronze) e a transformação consolidada com preenchimento de lacunas horárias (Prata).

In [1]:
import sys
from pathlib import Path
import pandas as pd
from IPython.display import display

# Garante resolução correta dos módulos a partir da raiz do projeto
BASE_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(BASE_DIR) not in sys.path:
    sys.path.insert(0, str(BASE_DIR))

from src.core.config import Config
print(f"Diretório base: {BASE_DIR}")

Diretório base: c:\Users\pedro\OneDrive\Documentos\Unifei\Topicos\sptrans-rain-impact


## 1. Camada Bronze: Ingestão de Dados de Chuva (API CEMADEN)

Dispara a ingestão para as estações pluviométricas de interesse (`stations_needed`), cobrindo a janela de coleta da SPTrans com margem de segurança. O arquivo de manifesto `_manifest.jsonl` rastreia os jobs concluídos para evitar re-downloads desnecessários.

In [ ]:
from src.pipelines.bronze.ingest_cemaden import main as ingest_cemaden_main

# Executa a ingestão Bronze do CEMADEN
bronze_status = ingest_cemaden_main([])
print(f"Status da ingestão Bronze: {bronze_status}")

bronze_csvs = list(Path(Config.CEMADEN_BRONZE_DIR).glob("station=*/req_*.csv"))
print(f"Total de arquivos CSV Bronze disponíveis: {len(bronze_csvs)}")

2026-10-01 13:14:25Z | INFO     | ingest_cemaden | Iniciando ingestão bronze do CEMADEN...
2026-10-01 13:14:45Z | INFO     | cemaden_client | Autenticação com CEMADEN realizada com sucesso.


## 2. Camada Prata: Limpeza e Geração do `rain_silver_export.csv`

Consolida as leituras pluviométricas brutas, descarta valores anômalos/sentinelas, preserva a granularidade exata das medições reais e gera linhas artificiais para horas sem dados (`is_missing = True`).

In [2]:
from src.pipelines.silver.build_cemaden_silver import main as build_cemaden_silver_main

# Executa o pipeline Silver gerando rain_silver_export.csv
silver_status = build_cemaden_silver_main([])
print(f"Status do pipeline Silver: {silver_status}")

2026-10-01 13:15:28Z | INFO     | build_cemaden_silver | Iniciando processamento Silver do CEMADEN...
2026-10-01 13:15:29Z | INFO     | build_cemaden_silver | Arquivo Silver gerado com sucesso: c:\Users\pedro\OneDrive\Documentos\Unifei\Topicos\sptrans-rain-impact\data\silver\cemaden\rain_silver_export.csv (918 linhas).


Status do pipeline Silver: 0


## 3. Inspeção dos Resultados (`rain_silver_export.csv`)

Carrega o dataset da Silver e inspeciona volumetria, distribuição de medições reais vs. faltantes e primeiras linhas.

In [3]:
silver_csv = Path(Config.CEMADEN_SILVER_DIR) / "rain_silver_export.csv"
df_silver = pd.read_csv(silver_csv)

print(f"Dimensões do dataset Silver: {df_silver.shape}")
print("\nContagem de registros (reais vs horas faltantes):")
print(df_silver["is_missing"].value_counts())

print("\nEstatísticas de precipitação (mm) para medições reais:")
display(df_silver.loc[~df_silver["is_missing"], "rain_mm"].describe().to_frame().T)

print("\nAmostra dos dados finais gerados:")
display(df_silver.head(10))

Dimensões do dataset Silver: (918, 4)

Contagem de registros (reais vs horas faltantes):
is_missing
False    749
True     169
Name: count, dtype: int64

Estatísticas de precipitação (mm) para medições reais:


,count,mean,std,min,25%,50%,75%,max
rain_mm,749.0,0.020267,0.064507,0.0,0.0,0.0,0.0,0.59



Amostra dos dados finais gerados:


,station_id,timestamp_sp,rain_mm,is_missing
0,355030802A,2026-09-29 00:10:00,0.0,False
1,355030802A,2026-09-29 01:10:00,0.0,False
2,355030802A,2026-09-29 02:10:00,0.0,False
3,355030802A,2026-09-29 03:10:00,0.0,False
4,355030802A,2026-09-29 04:10:00,0.0,False
5,355030802A,2026-09-29 05:10:00,0.0,False
6,355030802A,2026-09-29 06:10:00,0.0,False
7,355030802A,2026-09-29 07:10:00,0.0,False
8,355030802A,2026-09-29 08:10:00,0.0,False
9,355030802A,2026-09-29 09:10:00,0.0,False
